In [ ]:
# ============================================================
# ERAPave V2.3 — ROW 24 CANDIDATE SEARCH
# ============================================================
#
# Purpose:
#   Find promising E1, E2, E3, E4 combinations for Row 24
#   using the existing raw MLET solver and calibration data.
#
# IMPORTANT:
#   ERAPave/MLET is called with ALL locations simultaneously.
#
#   raw[0] = x = 0 cm  -> D0
#   raw[1] = x = 20 cm
#   raw[2] = x = 30 cm -> D30
#
#   SCI300 = D0 - D30
#
# The best candidate from this code is NOT automatically accepted.
# It must be verified with the actual ERAPave calculation.
# ============================================================


import numpy as np
import pandas as pd
from pathlib import Path
import importlib.util
import time


# ============================================================
# 1. LOAD RAW MLET MODULE
# ============================================================

MODULE_PATH = Path(
    "../data/ERAPave_MLET_V3_RAW_Benchmark.py"
)

spec = importlib.util.spec_from_file_location(
    "raw_mlet",
    MODULE_PATH
)

raw_mlet = importlib.util.module_from_spec(spec)
spec.loader.exec_module(raw_mlet)

print("Raw MLET module loaded.")


# ============================================================
# 2. ROW 24 INPUT DATA
# ============================================================

THICKNESS_CM = np.array([
    13.4246,
    42.3460,
    87.1116
], dtype=float)

POISSON = np.array([
    0.35,
    0.35,
    0.35,
    0.40
], dtype=float)

PRESSURE_KPA = 700.0

# Keep exactly the same locations used by the working code.
LOCATIONS_CM = np.array([
    0,
    20,
    30,
    60,
    90,
    120,
    150
], dtype=float)


# ============================================================
# 3. LOAD / CONTACT RADIUS
# ============================================================
#
# Keep these consistent with your existing ERAPave setup.
#
# If your existing V2.3 code already calculates RADIUS_CM,
# replace the next line with that exact value/calculation.
#

LOAD_KN = 50.0

# Circular contact radius:
# q = load / contact area
#
# load is converted from kN to kN consistently with pressure kPa.
#
# 1 kPa = 1 kN/m²
# radius is converted to cm.
#
# If your existing working code has a different RADIUS_CM,
# use that value instead.

LOAD_N = LOAD_KN * 1000.0
PRESSURE_PA = PRESSURE_KPA * 1000.0

AREA_M2 = LOAD_N / PRESSURE_PA
RADIUS_M = np.sqrt(AREA_M2 / np.pi)
RADIUS_CM = RADIUS_M * 100.0

print(f"Pressure = {PRESSURE_KPA:.1f} kPa")
print(f"Load     = {LOAD_KN:.1f} kN")
print(f"Radius   = {RADIUS_CM:.4f} cm")


# ============================================================
# 4. ROW 24 TARGETS
# ============================================================

TARGET_D0 = 2348.045
TARGET_SCI = 654.340
TARGET_D30 = 1693.705


print("\n" + "=" * 70)
print("ROW 24 TARGETS")
print("=" * 70)

print(f"Target D0     = {TARGET_D0:.3f} um")
print(f"Target D30    = {TARGET_D30:.3f} um")
print(f"Target SCI300 = {TARGET_SCI:.3f} um")


# ============================================================
# 5. EXACT WORKING RAW MLET FUNCTION
# ============================================================
#
# DO NOT change this to separate solver calls.
#
# The solver receives the complete LOCATIONS_CM array.
#

def raw_mlet_displacements(E):

    E = np.asarray(E, dtype=float)

    raw = raw_mlet.erapave_mlet(
        q_kpa=PRESSURE_KPA,
        radius_cm=RADIUS_CM,
        r_cm=LOCATIONS_CM,
        z_cm=np.zeros_like(LOCATIONS_CM),
        thickness_cm=THICKNESS_CM,
        modulus_mpa=E,
        poisson=POISSON,
        iteration=80
    )

    raw = np.asarray(raw, dtype=float).ravel()

    # Locations:
    # index 0 -> x=0 cm
    # index 1 -> x=20 cm
    # index 2 -> x=30 cm

    D0_raw = raw[0]
    D30_raw = raw[2]

    return D0_raw, D30_raw


# ============================================================
# 6. TEST THE RAW SOLVER ON ONE NORMAL POINT
# ============================================================

print("\n" + "=" * 70)
print("TESTING RAW MLET SOLVER")
print("=" * 70)

TEST_E = np.array([
    700.0,
    160.0,
    90.0,
    20.0
])

try:

    test_d0, test_d30 = raw_mlet_displacements(TEST_E)

    test_sci = test_d0 - test_d30

    print("Raw solver test successful.")
    print(f"E = {TEST_E}")

    print(f"Raw D0     = {test_d0:.3f}")
    print(f"Raw D30    = {test_d30:.3f}")
    print(f"Raw SCI300 = {test_sci:.3f}")

except Exception as e:

    print("\nERROR in raw MLET solver:")
    print(e)

    raise


# ============================================================
# 7. EXISTING ERAPAVE CALIBRATION DATA
# ============================================================
#
# These are REAL ERAPave calibration results.
#
# Columns:
# E1, E2, E3, E4, D0_ERAPAVE, D30_ERAPAVE
#
# SCI is calculated as:
#
# SCI300 = D0 - D30
#

CALIBRATION_DATA = np.array([

    # E1    E2    E3   E4     D0       D30
    [840,  195,  95,  20,   584.6,   359.5],
    [700,  160,  90,  20,   653.5,   383.6],
    [650,  180,  90,  20,   645.6,   375.3],
    [580,  180,  90,  20,   666.3,   377.2],
    [550,  200,  90,  20,   657.5,   370.2],
    [500,  220, 120,  20,   628.4,   337.1],
    [450,  180, 120,  20,   685.3,   333.4],
    [450,  250, 160,  20,   601.0,   306.1],
    [450,  180, 140,  20,   669.4,   337.7],
    [625,  100,  50,  25,   816.3,   466.8],
    [600,  120,  50,  20,   824.0,   487.8],

], dtype=float)


# ============================================================
# 8. CALIBRATION
# ============================================================
#
# We compare the raw MLET prediction against actual ERAPave.
#
# For each calibration point:
#
#   raw D0  -> actual ERAPave D0
#   raw D30 -> actual ERAPave D30
#
# We estimate global multiplicative correction factors.
#

print("\n" + "=" * 70)
print("CALIBRATING RAW MLET AGAINST ERAPAVE")
print("=" * 70)


calibration_results = []

for i, row in enumerate(CALIBRATION_DATA):

    E = row[:4]

    actual_d0 = row[4]
    actual_d30 = row[5]

    try:

        raw_d0, raw_d30 = raw_mlet_displacements(E)

        raw_sci = raw_d0 - raw_d30
        actual_sci = actual_d0 - actual_d30

        calibration_results.append([
            E[0],
            E[1],
            E[2],
            E[3],
            raw_d0,
            raw_d30,
            raw_sci,
            actual_d0,
            actual_d30,
            actual_sci
        ])

        print(
            f"{i+1:2d}: "
            f"E={E.astype(int)} | "
            f"raw D0={raw_d0:.2f} | "
            f"actual D0={actual_d0:.2f} | "
            f"raw D30={raw_d30:.2f} | "
            f"actual D30={actual_d30:.2f}"
        )

    except Exception as e:

        print(
            f"Calibration point {i+1} failed: "
            f"E={E} -> {e}"
        )


calibration_results = np.array(
    calibration_results,
    dtype=float
)


# ============================================================
# 9. GLOBAL CALIBRATION FACTORS
# ============================================================

raw_d0_values = calibration_results[:, 4]
raw_d30_values = calibration_results[:, 5]

actual_d0_values = calibration_results[:, 7]
actual_d30_values = calibration_results[:, 8]


# Multiplicative least-squares factor:
#
# actual ~= factor * raw
#

D0_FACTOR = (
    np.sum(raw_d0_values * actual_d0_values)
    /
    np.sum(raw_d0_values ** 2)
)

D30_FACTOR = (
    np.sum(raw_d30_values * actual_d30_values)
    /
    np.sum(raw_d30_values ** 2)
)


print("\n" + "-" * 70)
print("CALIBRATION FACTORS")
print("-" * 70)

print(f"D0  factor = {D0_FACTOR:.6f}")
print(f"D30 factor = {D30_FACTOR:.6f}")


# ============================================================
# 10. CALIBRATION PERFORMANCE
# ============================================================

cal_d0_pred = raw_d0_values * D0_FACTOR
cal_d30_pred = raw_d30_values * D30_FACTOR
cal_sci_pred = cal_d0_pred - cal_d30_pred

cal_d0_actual = actual_d0_values
cal_d30_actual = actual_d30_values
cal_sci_actual = cal_d0_actual - cal_d30_actual


def mean_absolute_percentage_error(actual, predicted):

    return np.mean(
        np.abs(
            (predicted - actual) / actual
        )
    ) * 100.0


print("\n" + "-" * 70)
print("CALIBRATION MAPE")
print("-" * 70)

print(
    f"D0     MAPE = "
    f"{mean_absolute_percentage_error(cal_d0_actual, cal_d0_pred):.3f}%"
)

print(
    f"D30    MAPE = "
    f"{mean_absolute_percentage_error(cal_d30_actual, cal_d30_pred):.3f}%"
)

print(
    f"SCI300 MAPE = "
    f"{mean_absolute_percentage_error(cal_sci_actual, cal_sci_pred):.3f}%"
)


# ============================================================
# 11. SEARCH RANGE FOR ROW 24
# ============================================================
#
# Row 24 has very large D0 = 2348 um.
#
# Therefore we search much softer moduli than Row 23.
#
# E1: 100 -> 700 MPa
# E2: 40  -> 200 MPa
# E3: 20  -> 120 MPa
# E4: 10,15,20,25,30 MPa
#
# Total:
#
# 25 * 17 * 11 * 5 = 23,375 candidates
#
# The raw solver is NOT called for every candidate.
#
# We first use the calibrated surrogate equations.
#

E1_VALUES = np.arange(
    100,
    701,
    25
)

E2_VALUES = np.arange(
    40,
    201,
    10
)

E3_VALUES = np.arange(
    20,
    121,
    10
)

E4_VALUES = np.array([
    10,
    15,
    20,
    25,
    30
], dtype=float)


print("\n" + "=" * 70)
print("ROW 24 SEARCH")
print("=" * 70)

print(f"E1 values: {len(E1_VALUES)}")
print(f"E2 values: {len(E2_VALUES)}")
print(f"E3 values: {len(E3_VALUES)}")
print(f"E4 values: {len(E4_VALUES)}")

TOTAL_CANDIDATES = (
    len(E1_VALUES)
    * len(E2_VALUES)
    * len(E3_VALUES)
    * len(E4_VALUES)
)

print(
    f"Total candidates = {TOTAL_CANDIDATES:,}"
)


# ============================================================
# 12. FAST RAW MLET SURROGATE
# ============================================================
#
# We need a fast approximation to search 23k combinations.
#
# We fit simple regression models from the calibration data:
#
# log(D0)  ~ log(E1,E2,E3,E4)
# log(D30) ~ log(E1,E2,E3,E4)
#
# This is only a candidate-ranking model.
#
# Actual ERAPave verification remains mandatory.
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor


X_cal = CALIBRATION_DATA[:, :4]

y_d0 = CALIBRATION_DATA[:, 4]
y_d30 = CALIBRATION_DATA[:, 5]


# Fit ExtraTrees directly on actual ERAPave outputs.
#
# This is more flexible than a simple multiplicative equation.

d0_model = ExtraTreesRegressor(
    n_estimators=500,
    random_state=42,
    max_features=1.0,
    min_samples_leaf=1,
    n_jobs=-1
)

d30_model = ExtraTreesRegressor(
    n_estimators=500,
    random_state=43,
    max_features=1.0,
    min_samples_leaf=1,
    n_jobs=-1
)


d0_model.fit(X_cal, y_d0)
d30_model.fit(X_cal, y_d30)


print("\nExtraTrees surrogate models fitted.")


# ============================================================
# 13. GENERATE ALL CANDIDATES
# ============================================================

grid = np.array(
    np.meshgrid(
        E1_VALUES,
        E2_VALUES,
        E3_VALUES,
        E4_VALUES,
        indexing="ij"
    )
).reshape(4, -1).T


print(
    f"Candidate matrix shape = {grid.shape}"
)


# ============================================================
# 14. PREDICT D0 AND D30 FOR ALL CANDIDATES
# ============================================================

start_time = time.time()

pred_d0 = d0_model.predict(grid)
pred_d30 = d30_model.predict(grid)

pred_sci = pred_d0 - pred_d30


# ============================================================
# 15. CALCULATE ERRORS
# ============================================================

d0_error_pct = (
    (pred_d0 - TARGET_D0)
    / TARGET_D0
    * 100.0
)

d30_error_pct = (
    (pred_d30 - TARGET_D30)
    / TARGET_D30
    * 100.0
)

sci_error_pct = (
    (pred_sci - TARGET_SCI)
    / TARGET_SCI
    * 100.0
)


# Absolute errors

abs_d0_error = np.abs(d0_error_pct)
abs_d30_error = np.abs(d30_error_pct)
abs_sci_error = np.abs(sci_error_pct)


# ============================================================
# 16. SCORING
# ============================================================
#
# Main targets:
#
#   D0
#   SCI300
#
# D30 is diagnostic.
#
# Therefore give D0 and SCI more importance.
#

score = (
    abs_d0_error
    +
    abs_sci_error
    +
    0.5 * abs_d30_error
)


results = pd.DataFrame({

    "E1_MPa": grid[:, 0],
    "E2_MPa": grid[:, 1],
    "E3_MPa": grid[:, 2],
    "E4_MPa": grid[:, 3],

    "D0_pred": pred_d0,
    "D30_pred": pred_d30,
    "SCI300_pred": pred_sci,

    "D0_error_pct": d0_error_pct,
    "D30_error_pct": d30_error_pct,
    "SCI300_error_pct": sci_error_pct,

    "score": score
})


results = results.sort_values(
    "score",
    ascending=True
).reset_index(drop=True)


elapsed = time.time() - start_time

print(
    f"\nSearch completed in {elapsed:.2f} seconds."
)


# ============================================================
# 17. TOP 30 CANDIDATES
# ============================================================

print("\n" + "=" * 100)
print("TOP 30 ROW 24 CANDIDATES")
print("=" * 100)

display(
    results.head(30).round(3)
)


# ============================================================
# 18. BEST CANDIDATE
# ============================================================

best = results.iloc[0]

print("\n" + "=" * 70)
print("BEST SURROGATE CANDIDATE")
print("=" * 70)

print(
    f"E1 = {best['E1_MPa']:.0f} MPa"
)

print(
    f"E2 = {best['E2_MPa']:.0f} MPa"
)

print(
    f"E3 = {best['E3_MPa']:.0f} MPa"
)

print(
    f"E4 = {best['E4_MPa']:.0f} MPa"
)

print()

print(
    f"Predicted D0     = "
    f"{best['D0_pred']:.2f} um"
)

print(
    f"Target D0        = "
    f"{TARGET_D0:.2f} um"
)

print(
    f"D0 error         = "
    f"{best['D0_error_pct']:+.2f}%"
)

print()

print(
    f"Predicted D30    = "
    f"{best['D30_pred']:.2f} um"
)

print(
    f"Target D30       = "
    f"{TARGET_D30:.2f} um"
)

print(
    f"D30 error        = "
    f"{best['D30_error_pct']:+.2f}%"
)

print()

print(
    f"Predicted SCI300 = "
    f"{best['SCI300_pred']:.2f} um"
)

print(
    f"Target SCI300    = "
    f"{TARGET_SCI:.2f} um"
)

print(
    f"SCI error        = "
    f"{best['SCI300_error_pct']:+.2f}%"
)

print()

print(
    f"Score            = "
    f"{best['score']:.3f}"
)


# ============================================================
# 19. CANDIDATES WITH D0 AND SCI WITHIN ±5%
# ============================================================

within_5 = results[
    (abs_d0_error <= 5.0)
    &
    (abs_sci_error <= 5.0)
].copy()


print("\n" + "=" * 100)
print("CANDIDATES WITH D0 AND SCI300 WITHIN ±5%")
print("=" * 100)

print(
    f"Number found = {len(within_5)}"
)

if len(within_5) > 0:

    display(
        within_5.head(30).round(3)
    )


# ============================================================
# 20. CANDIDATES WITH D0 AND SCI WITHIN ±2%
# ============================================================

within_2 = results[
    (abs_d0_error <= 2.0)
    &
    (abs_sci_error <= 2.0)
].copy()


print("\n" + "=" * 100)
print("CANDIDATES WITH D0 AND SCI300 WITHIN ±2%")
print("=" * 100)

print(
    f"Number found = {len(within_2)}"
)

if len(within_2) > 0:

    display(
        within_2.head(30).round(3)
    )

else:

    print(
        "No candidate satisfies both D0 and SCI300 "
        "within ±2% in the surrogate search."
    )


# ============================================================
# 21. TOP 10 FOR ACTUAL ERAPAVE VERIFICATION
# ============================================================
#
# IMPORTANT:
#
# Do NOT automatically accept the best surrogate candidate.
#
# We will take the best 10 candidates and verify them using
# the actual working raw MLET / ERAPave process.
#
# This is especially important because Row 23 showed that
# the surrogate can be inaccurate away from calibration points.
#

TOP_N = 10

top_candidates = results.head(TOP_N).copy()


print("\n" + "=" * 100)
print("TOP 10 CANDIDATES — VERIFY WITH ACTUAL ERAPAVE")
print("=" * 100)

display(
    top_candidates.round(3)
)


# ============================================================
# 22. VERIFY TOP 10 USING ACTUAL RAW MLET
# ============================================================
#
# This does NOT replace the actual ERAPave verification.
#
# It gives us a second check using the exact working solver.
#

verification = []


print("\n" + "=" * 100)
print("ACTUAL RAW MLET VERIFICATION")
print("=" * 100)


for i, (_, candidate) in enumerate(
    top_candidates.iterrows(),
    start=1
):

    E = np.array([
        candidate["E1_MPa"],
        candidate["E2_MPa"],
        candidate["E3_MPa"],
        candidate["E4_MPa"]
    ])

    try:

        raw_d0, raw_d30 = raw_mlet_displacements(E)

        # Apply calibration factors
        corrected_d0 = raw_d0 * D0_FACTOR
        corrected_d30 = raw_d30 * D30_FACTOR

        corrected_sci = (
            corrected_d0
            -
            corrected_d30
        )

        err_d0 = (
            (corrected_d0 - TARGET_D0)
            /
            TARGET_D0
            *
            100.0
        )

        err_d30 = (
            (corrected_d30 - TARGET_D30)
            /
            TARGET_D30
            *
            100.0
        )

        err_sci = (
            (corrected_sci - TARGET_SCI)
            /
            TARGET_SCI
            *
            100.0
        )

        verification.append([

            E[0],
            E[1],
            E[2],
            E[3],

            raw_d0,
            raw_d30,

            corrected_d0,
            corrected_d30,
            corrected_sci,

            err_d0,
            err_d30,
            err_sci,

            abs(err_d0)
            +
            abs(err_sci)
            +
            0.5 * abs(err_d30)

        ])

        print(
            f"{i:2d}: "
            f"E={E.astype(int)} | "
            f"D0={corrected_d0:.1f} "
            f"({err_d0:+.2f}%) | "
            f"D30={corrected_d30:.1f} "
            f"({err_d30:+.2f}%) | "
            f"SCI={corrected_sci:.1f} "
            f"({err_sci:+.2f}%)"
        )

    except Exception as e:

        print(
            f"{i:2d}: E={E.astype(int)} FAILED -> {e}"
        )


# ============================================================
# 23. VERIFICATION DATAFRAME
# ============================================================

verification = pd.DataFrame(
    verification,
    columns=[
        "E1_MPa",
        "E2_MPa",
        "E3_MPa",
        "E4_MPa",

        "D0_raw",
        "D30_raw",

        "D0_corrected",
        "D30_corrected",
        "SCI300_corrected",

        "D0_error_pct",
        "D30_error_pct",
        "SCI300_error_pct",

        "verification_score"
    ]
)


verification = verification.sort_values(
    "verification_score",
    ascending=True
).reset_index(drop=True)


# ============================================================
# 24. BEST ACTUAL-MLET VERIFIED CANDIDATE
# ============================================================

print("\n" + "=" * 100)
print("BEST CANDIDATE AFTER EXACT MLET VERIFICATION")
print("=" * 100)

if len(verification) > 0:

    best_verified = verification.iloc[0]

    print(
        f"E1 = {best_verified['E1_MPa']:.0f} MPa"
    )

    print(
        f"E2 = {best_verified['E2_MPa']:.0f} MPa"
    )

    print(
        f"E3 = {best_verified['E3_MPa']:.0f} MPa"
    )

    print(
        f"E4 = {best_verified['E4_MPa']:.0f} MPa"
    )

    print()

    print(
        f"D0  = {best_verified['D0_corrected']:.2f} um "
        f"({best_verified['D0_error_pct']:+.2f}%)"
    )

    print(
        f"D30 = {best_verified['D30_corrected']:.2f} um "
        f"({best_verified['D30_error_pct']:+.2f}%)"
    )

    print(
        f"SCI = {best_verified['SCI300_corrected']:.2f} um "
        f"({best_verified['SCI300_error_pct']:+.2f}%)"
    )


# ============================================================
# 25. SAVE RESULTS
# ============================================================

OUTPUT_FILE = Path(
    "/mnt/data/V2_3_Row24_Candidate_Search.xlsx"
)


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    results.to_excel(
        writer,
        sheet_name="All_Candidates",
        index=False
    )

    top_candidates.to_excel(
        writer,
        sheet_name="Top_10",
        index=False
    )

    within_5.to_excel(
        writer,
        sheet_name="Within_5pct",
        index=False
    )

    within_2.to_excel(
        writer,
        sheet_name="Within_2pct",
        index=False
    )

    verification.to_excel(
        writer,
        sheet_name="Exact_MLET_Verification",
        index=False
    )


print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print(
    f"Results saved to:\n{OUTPUT_FILE}"
)

print()
print(
    "IMPORTANT: The candidate is NOT yet the final Row 24 label."
)

print(
    "Run the best 1–3 candidates through the actual ERAPave "
    "calculation and compare:"
)

print(
    "D0 = x=0 cm"
)

print(
    "D30 = x=30 cm"
)

print(
    "SCI300 = D0 - D30"
)

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\mnt\\data\\ERAPave_MLET_V3_RAW_Benchmark.py'